In [9]:
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Union
from copy import deepcopy
from collections import defaultdict
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
import itertools
import optuna

@dataclass
class Coil:
    """Барабан с неизолированной жилой"""
    id: str
    length: float
    max_payoff: Optional[float] = None  # максимальная длина единовременной смотки
    used: float = 0
    
    @property
    def available(self):
        return self.length - self.used

@dataclass
class PreInsulatedCore:
    """Готовая изолированная жила (может быть использована частично)"""
    id: str
    color: str
    length: float
    used: bool = False

@dataclass
class Cable:
    """Заказ на кабель"""
    name: str
    length: float
    cores: int
    colors: List[str]
    priority: int = 1
    
    def __post_init__(self):
        if len(self.colors) != self.cores:
            raise ValueError(f"Цветов {len(self.colors)} != жил {self.cores}")

@dataclass
class Allocation:
    """Распределение из сырья (барабанов)"""
    cable_name: str
    color: str
    coil_id: str
    length: float

@dataclass
class PreUsed:
    """Использованная готовая жила (или её часть)"""
    cable_name: str
    color: str
    pre_id: str
    length: float
    original_length: float  # для справки

# ПЛАН МОДЕРНИЗАЦИИ — Length Helper

## Что делает система (новое понимание из ТЗ)

**Задача:** по Excel-заявке спланировать производство кабеля на двух этапах:
1. **Изолирование** — намотать цветную изоляцию на голую жилу (ТПЖ). Ограничение: ≤ 4500 м за один проход.
2. **Скрутка** — скрутить несколько цветных жил в кабель. Ограничение: ≤ 2100 м за один проход.

**Выход:** кабель режется отрезками по кабельному журналу и наматывается на барабаны (размеры 10/12/14).

---

## Ключевые новые концепции (vs старый код)

| Старый код | Новый код | Смысл |
|---|---|---|
| `Coil` | `RawWire` (ТПЖ) | Барабан с голой жилой, источник для изолирования |
| `PreInsulatedCore` | `InsulatedCore` | Уже готовая изолированная жила на складе |
| `Cable` | `CableOrder` | Заказ + **кабельный журнал** (список отрезков!) |
| нет | `TwistingBatch` | Партия скрутки: группа отрезков журнала ≤ 2100 м |
| нет | `InsulationRun` | Проход изолирования: цвет + источник + длина |
| нет | `DrumAssignment` | Назначение выходного барабана под отрезки журнала |
| нет | `DrumCapacity` | Ёмкость барабана 10/12/14 для каждой марки |
| нет | `ProcessParams` | Ограничения: max изоляция / скрутка, спайка и т.д. |

---

## Входной Excel (структура)

**Лист "Заказы":**
```
Марка кабеля | Длина заказа, м | Кабельный журнал (через запятую)
```
- Если журнала нет — делим на отрезки по минимальной строительной длине (450 м)

**Лист "П/Ф" (полуфабрикаты):**
```
Наименование | Длина, м
ТПЖ 2,5ок    | 9000      ← голая жила (источник для изолирования)
Синяя 2,5ок  | 6000      ← уже изолированная жила на складе
```

**Лист "Барабаны" (ёмкость):**
```
Марка кабеля | Барабан 10 | Барабан 12 | Барабан 14
```

**Лист "Параметры":**
```
Намотка на изолировании  | 4500 м
Намотка скрученной заготовки | 2100 м
Минимальная строительная длина | 450 м
Можно ли спаивать жилы   | нет
```

**Лист "Марки" (состав кабелей):**
```
Марка кабеля | Цвет жилы 1 | Цвет жилы 2 | ...
```

---

## Алгоритм (пошагово)

### Шаг 1. Парсинг входного Excel
- Читаем все листы → создаём объекты `CableOrder`, `RawWire`, `InsulatedCore`, `DrumCapacity`, `ProcessParams`

### Шаг 2. Bin packing журнала → партии скрутки (TwistingBatch)
- Для каждого заказа: группируем отрезки кабельного журнала в партии с суммой ≤ 2100 м
- Цель: минимальное количество партий (= меньше перенастроек оборудования)
- Алгоритм: First Fit Decreasing (FFD)
- Если журнала нет: бьём заказ на отрезки по 2100 м

### Шаг 3. Распределение жил под каждую партию скрутки
- Для каждой (партия × цвет): нужна одна непрерывная жила длиной = длина партии
- Приоритет 1: взять из склада готовых `InsulatedCore` (по цвету)
  - Если жила на складе ≥ нужной длины → отрезаем, остаток возвращаем на склад
  - Если жила на складе < нужной → **нельзя использовать** (нет спайки!)
- Приоритет 2: запланировать изолирование из ТПЖ
  - Берём отрезок из `RawWire` ≤ 4500 м → создаём `InsulationRun`
  - Если нужная длина > 4500 м → ОШИБКА (невозможно без спайки)

### Шаг 4. Назначение выходных барабанов (DrumAssignment)
- Для каждой партии скрутки: её отрезки журнала назначаем на барабаны
- Подбираем наименьший барабан (10 < 12 < 14), куда помещается сумма текущей группы
- Bin packing: пакуем соседние отрезки на один барабан, если сумма ≤ ёмкости барабана
- Стратегия: минимальное количество барабанов = экономия

### Шаг 5. Составление производственных инструкций
- **Инструкция изолирования**: цвет → источник ТПЖ (id, было/стало) → длина прогона
- **Инструкция скрутки**: № партии → марка кабеля → длина → жилы (откуда каждая)
- **Инструкция намотки**: отрезки журнала → барабан (размер, № п/п)

### Шаг 6. Экспорт в Excel

---

## Исправляемые баги старого кода

1. **Счётчик покрытых жил** — считал записи `PreUsed`, а не целые жилы. Новая модель исключает проблему: одна жила = один `InsulationRun` или один `InsulatedCore`.
2. **Неполный откат** — при провале сборки жилы из нескольких кусков созданные "остатки" не удалялись. В новой модели жила либо полностью покрывается одним куском, либо нет (нет спайки).
3. **Optuna без эффекта** — веса не влияли на алгоритм, только на оценку. Убираем Optuna, заменяем детерминированным алгоритмом с FFD.
4. **Повторное использование барабана при max_payoff** — в новой модели ограничение 4500 м задаётся явно в `ProcessParams`.
5. **Опечатки в цветах** — нормализуем цвета при парсинге входного файла (strip + lower).

---

## Структура выходного Excel

| Лист | Содержимое |
|---|---|
| 1. Исходные данные | Заказы, П/Ф, параметры |
| 2. Инструкция изолирования | Цвет, ТПЖ-источник, длина прогона, было/стало |
| 3. Инструкция скрутки | Партия, марка, длина, жилы (источники) |
| 4. Инструкция намотки | Отрезки журнала → барабаны |
| 5. Остатки ТПЖ | Что осталось на каждом барабане |
| 6. Остатки изолированных жил | Неиспользованные п/ф |
| 7. Ошибки | Нехватка материала, превышения ограничений |


In [10]:
def allocate_cores_optimized(
    coils: List[Coil],
    cables: List[Cable],
    preinsulated_cores: Optional[List[PreInsulatedCore]] = None,
    allow_splitting: bool = True,
    min_segment_length: float = 100.0,
    optimize_by_color: bool = True,
    strategy: str = 'minimize_waste',
    weights: Optional[Dict[str, float]] = None
) -> Tuple[List[Allocation], List[PreUsed], List[str], Dict, List[PreInsulatedCore]]:
    """
    Распределяет барабаны и готовые жилы для изготовления заказов.
    Готовые жилы можно резать и комбинировать.
    Возвращает:
        allocations_из_сырья,
        used_pre_cores (список использованных готовых жил с указанием отрезков),
        errors,
        stats,
        remaining_pre_cores (остатки готовых жил после распределения)
    """
    if weights is None:
        weights = {
            'error_penalty': 10000,
            'waste_penalty': 10,
            'transition_penalty': 100,
            'segment_penalty': 50,
            'pre_bonus': 200
        }
    
    allocations = []
    used_pre = []
    errors = []
    coils = deepcopy(coils)
    # Работаем с копией списка готовых жил, который будем модифицировать
    remaining_pre = deepcopy(preinsulated_cores) if preinsulated_cores else []
    
    stats = {
        'color_transitions': 0,
        'total_segments': 0,
        'multi_segment_cores': 0,
        'total_waste': sum(c.available for c in coils),
        'used_preinsulated': 0,
        'strategy': strategy
    }
    
    # ---- 1. Формирование потребностей по каждому заказу ----
    # Для каждого заказа: словарь {цвет: необходимое_количество_жил}
    requirements = []
    for cable in cables:
        color_counts = defaultdict(int)
        for color in cable.colors:
            color_counts[color] += 1
        requirements.append({
            'cable_name': cable.name,
            'length': cable.length,
            'color_counts': dict(color_counts),
            'priority': cable.priority
        })
    
    # Сортируем заказы по стратегии (для порядка обработки)
    if strategy == 'minimize_waste':
        requirements.sort(key=lambda r: (-r['length'], r['cable_name']))
    elif strategy == 'minimize_transitions':
        requirements.sort(key=lambda r: list(r['color_counts'].keys()))
    else:  # balanced
        requirements.sort(key=lambda r: (list(r['color_counts'].keys()), -r['length']))
    
    # ---- 2. Использование готовых жил (с возможностью резать) ----
    # Для каждого заказа обрабатываем цвета
    for req in requirements:
        cable_name = req['cable_name']
        target_length = req['length']
        color_counts = req['color_counts'].copy()  # будем уменьшать
        
        for color, need in list(color_counts.items()):
            for _ in range(need):
                # Пытаемся покрыть одну жилу длины target_length из готовых жил цвета color
                length_needed = target_length
                temp_used = []  # временные использования для этой жилы
                
                # Сортируем доступные готовые жилы этого цвета по убыванию длины (жадная стратегия)
                available = [p for p in remaining_pre if not p.used and p.color == color]
                available.sort(key=lambda p: -p.length)
                
                while length_needed > 0 and available:
                    # Берём самую длинную
                    p = available[0]
                    if p.length >= length_needed:
                        # Отрезаем нужный кусок
                        temp_used.append(PreUsed(
                            cable_name=cable_name,
                            color=color,
                            pre_id=p.id,
                            length=length_needed,
                            original_length=p.length
                        ))
                        # Остаток: создаём новую готовую жилу
                        if p.length - length_needed >= 1e-6:  # если остаток больше погрешности
                            new_id = f"{p.id}-остаток"
                            remaining_pre.append(PreInsulatedCore(
                                id=new_id,
                                color=color,
                                length=p.length - length_needed,
                                used=False
                            ))
                        # Помечаем исходную как использованную
                        p.used = True
                        length_needed = 0
                        break
                    else:
                        # Берём целиком
                        temp_used.append(PreUsed(
                            cable_name=cable_name,
                            color=color,
                            pre_id=p.id,
                            length=p.length,
                            original_length=p.length
                        ))
                        length_needed -= p.length
                        p.used = True
                        # Обновляем список available (удаляем использованную)
                        available.pop(0)
                
                if length_needed <= 1e-6:
                    # Жила полностью собрана из готовых
                    used_pre.extend(temp_used)
                    stats['used_preinsulated'] += len(temp_used)
                    # Уменьшаем потребность в color_counts (но мы уже итерируем, поэтому просто удалим из списка? 
                    # Мы обрабатываем каждую жилу по очереди, поэтому просто продолжаем)
                    # Ничего не делаем, потребность уже учтена через цикл for _ in range(need)
                else:
                    # Не удалось полностью собрать из готовых, остаток length_needed > 0
                    # Откатываем временные использования (помечаем жилы как неиспользованные)
                    for tu in temp_used:
                        # Найти исходную жилу и снять пометку used
                        for p in remaining_pre:
                            if p.id == tu.pre_id:
                                p.used = False
                                break
                    # И удалить созданные остатки? Мы не создавали остатки, так как не дошли до отрезания.
                    # Созданные остатки были бы в случае p.length >= length_needed, но тогда length_needed стал бы 0.
                    # Значит, в этой ветке мы брали только целые жилы, поэтому новых не создавали.
                    
                    # Теперь эта жила будет производиться из сырья. Добавляем задачу позже.
                    # Пока просто запомним, что нужно добавить задачу.
                    # Мы сделаем это на следующем этапе, собрав все неудовлетворённые потребности.
                    # Но поскольку мы уже уменьшили need в цикле, а жила не произведена, нужно её куда-то записать.
                    # Проще всего сформировать список задач после обработки готовых жил.
                    # Поэтому мы не будем здесь уменьшать need сразу, а будем накапливать неудовлетворённые жилы.
                    pass
    
    # ---- 3. Сбор неудовлетворённых потребностей (задачи на производство из сырья) ----
    # После использования готовых жил у нас остались необработанными те жилы, для которых не хватило готовых.
    # Нужно заново пройти по требованиям и вычесть то, что уже покрыто used_pre.
    
    # Сначала построим множество покрытых (cable_name, color) с учётом количества.
    covered = defaultdict(lambda: defaultdict(int))
    for up in used_pre:
        covered[up.cable_name][up.color] += 1
    
    tasks = []
    for req in requirements:
        cable_name = req['cable_name']
        length = req['length']
        for color, need in req['color_counts'].items():
            already = covered[cable_name][color]
            for _ in range(need - already):
                tasks.append({
                    'cable_name': cable_name,
                    'color': color,
                    'length': length,
                    'priority': req['priority']
                })
    
    # Сортировка задач по стратегии (как в оригинале)
    if strategy == 'minimize_waste':
        tasks.sort(key=lambda t: (-t['length'], t['color']))
    elif strategy == 'minimize_transitions':
        tasks.sort(key=lambda t: t['color'])
    elif strategy == 'balanced':
        tasks.sort(key=lambda t: (t['color'], -t['length']))
    
    if optimize_by_color:
        tasks.sort(key=lambda t: t['color'])
    
    # Подсчет переходов цветов при производстве (только для новых жил)
    prev_color = None
    for task in tasks:
        if prev_color and prev_color != task['color']:
            stats['color_transitions'] += 1
        prev_color = task['color']
    
    # ---- 4. Производство новых жил из сырья (барабанов) с учётом max_payoff ----
    for task in tasks:
        cable_name = task['cable_name']
        color = task['color']
        required_length = task['length']
        allocated = False
        
        # Функция поиска одного барабана, который может дать всю длину с учётом max_payoff
        def find_single_coil(needed):
            candidates = []
            for c in coils:
                if c.available >= needed:
                    if c.max_payoff is None or c.max_payoff >= needed:
                        candidates.append(c)
            if not candidates:
                return None
            if strategy == 'minimize_waste':
                return min(candidates, key=lambda c: c.available)
            else:
                return candidates[0]  # первый подходящий (по порядку)
        
        coil = find_single_coil(required_length)
        if coil:
            allocations.append(Allocation(
                cable_name=cable_name,
                color=color,
                coil_id=coil.id,
                length=required_length
            ))
            coil.used += required_length
            stats['total_segments'] += 1
            allocated = True
            continue
        
        # Если не нашли одного барабана, пробуем собрать из нескольких
        if not allow_splitting:
            errors.append(
                f"⚠️ {cable_name}, {color}: нужен целый отрезок {required_length:.0f}м, "
                f"но нет подходящего барабана"
            )
            continue
        
        remaining = required_length
        temp_allocations = []
        
        # Сортируем барабаны по убыванию доступного (жадно)
        for coil in sorted(coils, key=lambda c: c.available, reverse=True):
            if coil.available <= 0 or remaining <= 0:
                continue
            
            # Сколько можем взять с этого барабана за один раз с учётом max_payoff
            max_take = coil.available
            if coil.max_payoff is not None:
                max_take = min(max_take, coil.max_payoff)
            
            take = min(max_take, remaining)
            
            # Проверка на минимальную длину отрезка (кроме последнего)
            if take < min_segment_length and take < remaining:
                continue  # этот кусок слишком мал для промежуточного, пробуем следующий барабан
            
            temp_allocations.append(Allocation(
                cable_name=cable_name,
                color=color,
                coil_id=coil.id,
                length=take
            ))
            coil.used += take
            remaining -= take
            
            # Если после взятия на барабане ещё осталось, но мы не можем взять ещё один кусок из-за max_payoff,
            # то продолжаем цикл, возможно, следующий барабан подойдёт. Но если остаток на этом барабане большой,
            # а remaining ещё положителен, мы можем вернуться к этому же барабану позже, если разрешено многократное использование.
            # В текущей реализации мы проходим по барабанам один раз, поэтому может быть неоптимально.
            # Упростим: если после взятия остаток на барабане >0 и remaining>0, и мы не превысили max_payoff (т.е. взяли меньше, чем могли),
            # то мы можем попробовать взять ещё с этого же барабана, но для этого нужно перезапустить цикл или использовать while.
            # Сделаем проще: если take == max_take и coil.available > 0, то мы можем попробовать ещё раз с этим же барабаном в следующей итерации.
            # Но так как мы сортируем по убыванию, этот барабан будет в начале списка, и мы можем пропустить его, если не обновляем сортировку.
            # Лучше реализовать цикл, который пока remaining>0, ищет лучший доступный источник.
        
        # После цикла проверяем, удалось ли покрыть всю длину
        if remaining > 0:
            errors.append(
                f"⚠️ {cable_name}, {color}: не хватает {remaining:.0f}м "
                f"(мин. отрезок: {min_segment_length:.0f}м)"
            )
            # Откат
            for alloc in temp_allocations:
                for coil in coils:
                    if coil.id == alloc.coil_id:
                        coil.used -= alloc.length
        else:
            allocations.extend(temp_allocations)
            stats['total_segments'] += len(temp_allocations)
            if len(temp_allocations) > 1:
                stats['multi_segment_cores'] += 1
    
    # Пересчёт отходов
    stats['total_waste'] = sum(c.available for c in coils)
    
    # Удаляем из remaining_pre те, что помечены used
    remaining_pre = [p for p in remaining_pre if not p.used]
    
    return allocations, used_pre, errors, stats, remaining_pre

In [11]:
def optimize_with_optuna(
    coils: List[Coil],
    cables: List[Cable],
    preinsulated_cores: Optional[List[PreInsulatedCore]] = None,
    allow_splitting: bool = True,
    min_segment_length: float = 100.0,
    n_trials: int = 50
) -> Tuple[Dict, float, Dict]:
    """
    Использует Optuna для поиска лучших весов в скоринговой функции.
    Возвращает: лучшие веса, лучшее значение целевой функции, лучшую статистику.
    """
    
    def objective(trial):
        # Предлагаем веса в разумных диапазонах
        weights = {
            'error_penalty': trial.suggest_float('error_penalty', 5000, 20000, log=True),
            'waste_penalty': trial.suggest_float('waste_penalty', 1, 50, log=True),
            'transition_penalty': trial.suggest_float('transition_penalty', 10, 500, log=True),
            'segment_penalty': trial.suggest_float('segment_penalty', 10, 200, log=True),
            'pre_bonus': trial.suggest_float('pre_bonus', 50, 500, log=True)
        }
        
        # Пробуем разные стратегии (выбираем одну фиксированную или тоже оптимизируем)
        # Для простоты возьмём balanced, но можно и стратегию сделать параметром
        strategy = 'balanced'
        
        allocs, used_pre, errors, stats, remaining = allocate_cores_optimized(
            coils=coils,
            cables=cables,
            preinsulated_cores=preinsulated_cores,
            allow_splitting=allow_splitting,
            min_segment_length=min_segment_length,
            optimize_by_color=True,
            strategy=strategy,
            weights=weights
        )
        
        # Целевая функция: минимизируем взвешенную сумму
        score = 0
        score -= len(errors) * weights['error_penalty']
        score -= stats['total_waste'] * weights['waste_penalty']
        score -= stats['color_transitions'] * weights['transition_penalty']
        score -= stats['multi_segment_cores'] * weights['segment_penalty']
        score += stats['used_preinsulated'] * weights['pre_bonus']
        
        return -score  # Optuna минимизирует, поэтому возвращаем отрицательный score (чем меньше ошибок/отходов, тем лучше)
    
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    
    best_weights = study.best_params
    best_value = study.best_value
    
    # Посчитать результат с лучшими весами
    allocs, used_pre, errors, stats, remaining = allocate_cores_optimized(
        coils=coils,
        cables=cables,
        preinsulated_cores=preinsulated_cores,
        allow_splitting=allow_splitting,
        min_segment_length=min_segment_length,
        optimize_by_color=True,
        strategy='balanced',
        weights=best_weights
    )
    
    return best_weights, best_value, {'allocs': allocs, 'used_pre': used_pre, 'errors': errors, 'stats': stats, 'remaining_pre': remaining}

In [12]:
def optimize_best_solution(
    coils: List[Coil],
    cables: List[Cable],
    preinsulated_cores: Optional[List[PreInsulatedCore]] = None,
    allow_splitting: bool = True,
    min_segment_length: float = 100.0
) -> Tuple[List[Allocation], List[PreUsed], List[str], Dict, pd.DataFrame, List[PreInsulatedCore]]:
    """
    Автоматически находит лучшее решение, пробуя разные стратегии.
    Возвращает: (allocs, used_pre, errors, stats, comparison_df, remaining_pre)
    """
    strategies = [
        ('minimize_waste', True, 'Минимизация отходов'),
        ('minimize_transitions', True, 'Минимизация переходов'),
        ('balanced', True, 'Сбалансированная'),
        ('minimize_waste', False, 'Без оптимизации по цветам'),
    ]
    
    results = []
    
    for strategy, optimize_by_color, description in strategies:
        allocs, used_pre, errs, stats, remaining = allocate_cores_optimized(
            coils=coils,
            cables=cables,
            preinsulated_cores=preinsulated_cores,
            allow_splitting=allow_splitting,
            min_segment_length=min_segment_length,
            optimize_by_color=optimize_by_color,
            strategy=strategy
        )
        
        # Оценка решения (с фиксированными весами)
        score = 0
        error_penalty = 10000
        waste_penalty = 10
        transition_penalty = 100
        segment_penalty = 50
        pre_bonus = 200
        
        score -= len(errs) * error_penalty
        score -= stats['total_waste'] * waste_penalty
        score -= stats['color_transitions'] * transition_penalty
        score -= stats['multi_segment_cores'] * segment_penalty
        score += stats['used_preinsulated'] * pre_bonus
        
        results.append({
            'strategy': description,
            'allocations': allocs,
            'used_pre': used_pre,
            'errors': errs,
            'stats': stats,
            'remaining_pre': remaining,
            'score': score,
            'waste': stats['total_waste'],
            'transitions': stats['color_transitions'],
            'segments': stats['multi_segment_cores'],
            'used_pre_count': stats['used_preinsulated'],
            'error_count': len(errs)
        })
    
    best = max(results, key=lambda r: r['score'])
    
    # Таблица сравнения
    comparison_data = []
    scores = [r['score'] for r in results]
    min_score, max_score = min(scores), max(scores)
    for r in sorted(results, key=lambda x: x['score'], reverse=True):
        is_best = r == best
        rating = 0
        if max_score > min_score:
            rating = int((r['score'] - min_score) / (max_score - min_score) * 100)
        comparison_data.append({
            'Стратегия': r['strategy'] + (' ⭐' if is_best else ''),
            'Ошибок': r['error_count'],
            'Отходы (м)': int(r['waste']),
            'Переходы': r['transitions'],
            'Составных жил': r['segments'],
            'Исп. готовых': r['used_pre_count'],
            'Рейтинг (%)': rating
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    return best['allocations'], best['used_pre'], best['errors'], best['stats'], comparison_df, best['remaining_pre']

In [13]:
def export_to_excel(filename, coils, cables, allocations, used_pre, comparison_df=None, errors=None, remaining_pre=None):
    """Экспорт с листами: исходные, сравнение, инструкция, остатки барабанов, готовые жилы, ошибки"""
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # Лист 1: Исходные данные
        df_coils = pd.DataFrame([{
            'Барабан': c.id,
            'Длина (м)': int(c.length),
            'Max смотка (м)': int(c.max_payoff) if c.max_payoff else '-'
        } for c in coils])
        df_cables = pd.DataFrame([{
            'Заказ': c.name,
            'Длина (м)': int(c.length),
            'Жильность': f"{c.cores} жил",
            'Цвета': ', '.join(c.colors),
            'Приоритет': c.priority
        } for c in cables])
        
        df_coils.to_excel(writer, sheet_name='1. Исходные данные', startrow=1, index=False)
        df_cables.to_excel(writer, sheet_name='1. Исходные данные', 
                          startrow=len(df_coils) + 5, index=False)
        
        ws1 = writer.sheets['1. Исходные данные']
        ws1['A1'] = 'БАРАБАНЫ С ЖИЛОЙ'
        ws1['A1'].font = Font(bold=True, size=12)
        row_cables_header = len(df_coils) + 5 + 1
        ws1[f'A{row_cables_header}'].value = 'ЗАКАЗЫ НА КАБЕЛЬ'
        ws1[f'A{row_cables_header}'].font = Font(bold=True, size=12)
        
        # Лист 2: Сравнение стратегий
        if comparison_df is not None:
            comparison_df.to_excel(writer, sheet_name='2. Оптимизация', startrow=1, index=False)
            ws2 = writer.sheets['2. Оптимизация']
            ws2['A1'] = 'СРАВНЕНИЕ СТРАТЕГИЙ ОПТИМИЗАЦИИ'
            ws2['A1'].font = Font(bold=True, size=14)
        
        # Лист 3: Инструкция (объединяем готовые и новые)
        steps = []
        # Сначала готовые жилы
        for up in used_pre:
            cable = next((c for c in cables if c.name == up.cable_name), None)
            steps.append({
                'Цвет': up.color,
                'Заказ': up.cable_name,
                'Длина заказа (м)': int(cable.length) if cable else '-',
                'Изолировать (м)': int(up.length),
                'Источник': f"Готовая {up.pre_id} (было {int(up.original_length)} м)",
                'Было на барабане (м)': '-',
                'Осталось на барабане (м)': '-'
            })
        # Затем новые из сырья
        coil_balances = {coil.id: coil.length for coil in coils}
        for alloc in allocations:
            cable = next((c for c in cables if c.name == alloc.cable_name), None)
            balance_before = coil_balances.get(alloc.coil_id, 0)
            coil_balances[alloc.coil_id] = balance_before - alloc.length
            balance_after = coil_balances[alloc.coil_id]
            steps.append({
                'Цвет': alloc.color,
                'Заказ': alloc.cable_name,
                'Длина заказа (м)': int(cable.length) if cable else '-',
                'Изолировать (м)': int(alloc.length),
                'Источник': f'Барабан {alloc.coil_id}',
                'Было на барабане (м)': int(balance_before),
                'Осталось на барабане (м)': int(balance_after)
            })
        
        # Сортируем по цвету для наглядности
        steps.sort(key=lambda x: x['Цвет'])
        df_instr = pd.DataFrame(steps)
        df_instr.insert(0, '№', range(1, len(df_instr)+1))
        df_instr.to_excel(writer, sheet_name='3. Инструкция', startrow=1, index=False)
        ws3 = writer.sheets['3. Инструкция']
        ws3['A1'] = 'ПОШАГОВАЯ ПРОИЗВОДСТВЕННАЯ ИНСТРУКЦИЯ'
        ws3['A1'].font = Font(bold=True, size=14)
        
        # Лист 4: Остатки на барабанах
        used_by_coil = defaultdict(float)
        for a in allocations:
            used_by_coil[a.coil_id] += a.length
        remainders = []
        for coil in coils:
            used = used_by_coil.get(coil.id, 0)
            remain = coil.length - used
            pct = (used / coil.length * 100) if coil.length else 0
            remainders.append({
                'Барабан': coil.id,
                'Всего (м)': coil.length,
                'Использовано (м)': used,
                'Остаток (м)': remain,
                'Использование (%)': f"{pct:.1f}%"
            })
        df_rem = pd.DataFrame(remainders)
        df_rem.to_excel(writer, sheet_name='4. Остатки', startrow=1, index=False)
        ws4 = writer.sheets['4. Остатки']
        ws4['A1'] = 'ОСТАТКИ НА БАРАБАНАХ'
        ws4['A1'].font = Font(bold=True, size=12)
        
        # Лист 5: Остатки готовых жил
        if remaining_pre:
            df_pre_rem = pd.DataFrame([{
                'ID': p.id,
                'Цвет': p.color,
                'Длина (м)': int(p.length)
            } for p in remaining_pre])
            df_pre_rem.to_excel(writer, sheet_name='5. Готовые жилы (остатки)', startrow=1, index=False)
            ws5 = writer.sheets['5. Готовые жилы (остатки)']
            ws5['A1'] = 'НЕИСПОЛЬЗОВАННЫЕ ОСТАТКИ ГОТОВЫХ ЖИЛ'
            ws5['A1'].font = Font(bold=True, size=12)
        
        # Лист 6: Ошибки
        if errors and len(errors) > 0:
            df_errors = pd.DataFrame({'Ошибка': errors})
            df_errors.to_excel(writer, sheet_name='6. Ошибки', startrow=1, index=False)
            ws6 = writer.sheets['6. Ошибки']
            ws6['A1'] = 'СПИСОК ОШИБОК'
            ws6['A1'].font = Font(bold=True, size=14)
    
    # Форматирование (как в оригинале)
    wb = load_workbook(filename)
    thin_border = Border(left=Side(style='thin'), right=Side(style='thin'),
                         top=Side(style='thin'), bottom=Side(style='thin'))
    header_fill = PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid')
    
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for column in ws.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            ws.column_dimensions[column_letter].width = adjusted_width
        
        for cell in ws[2]:
            if cell.value:
                cell.font = Font(bold=True, size=11)
                cell.fill = header_fill
                cell.alignment = Alignment(horizontal='center', vertical='center')
                cell.border = thin_border
        
        for row in ws.iter_rows():
            for cell in row:
                if cell.value:
                    cell.border = thin_border
                    cell.alignment = Alignment(vertical='center')
    
    # Цветная подсветка в инструкции
    if '3. Инструкция' in wb.sheetnames:
        ws_instr = wb['3. Инструкция']
        color_fills = {
            'ж/з': PatternFill(start_color='FFFF99', end_color='FFFF99', fill_type='solid'),
            'синий': PatternFill(start_color='ADD8E6', end_color='ADD8E6', fill_type='solid'),
            'чёрный': PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid'),
            'коричневый': PatternFill(start_color='DEB887', end_color='DEB887', fill_type='solid'),
            'натуральный': PatternFill(start_color='F5F5DC', end_color='F5F5DC', fill_type='solid'),
        }
        for row in ws_instr.iter_rows(min_row=3):
            color_cell = row[1]  # колонка B
            if color_cell.value and color_cell.value in color_fills:
                fill = color_fills[color_cell.value]
                for cell in row:
                    if cell.value:
                        cell.fill = fill
    
    wb.save(filename)

In [ ]:
# Данные
pre_cores = [
    # PreInsulatedCore(id="ГЖ-001", color="ж/з", length=400),
    # PreInsulatedCore(id="ГЖ-002", color="синий", length=400),
    # PreInsulatedCore(id="ГЖ-003", color="коричневый", length=400),
    # PreInsulatedCore(id="ГЖ-004", color="чёрный", length=400),
]

coils = [
    Coil(id='Барабан-1', length=2620, max_payoff=5000),
    Coil(id='Барабан-2', length=2950, max_payoff=5000),
    Coil(id='Барабан-3', length=2400, max_payoff=5000),
    Coil(id='Барабан-4', length=2400, max_payoff=5000),
    Coil(id='Барабан-5', length=2400, max_payoff=5000),
]



cables = [
    
    Cable(name='5х35мк', length=1000, cores=5, colors=['ж/з','синий','чёрный','коричневый','натуральный']),
    Cable(name='3х35мк', length=800, cores=3, colors=['ж/з','синий','коричневый']),
    Cable(name='4х35мк', length=1200, cores=4, colors=['синий','черный','коричневый','натуральный']),
          ]

# Оптимизация
allocs, used_pre, errors, stats, comp_df, remaining_pre = optimize_best_solution(
    coils, cables, preinsulated_cores=pre_cores, allow_splitting=True, min_segment_length=300
)

# Экспорт
export_to_excel("result.xlsx", coils, cables, allocs, used_pre, comp_df, errors, remaining_pre)

print("Готово!")

Готово!
